# 🧩 Mini-Lab: CSV and JSON: stdlib vs pandas, and How to Choose

**Module: Real Data: Paths, CSV, JSON, pandas, NumPy** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Read and write** CSV files with both the `csv` stdlib module and `pandas`
2. **Read and write** JSON and JSON Lines (JSONL) with both `json` stdlib and `pandas`
3. **Choose** between stdlib and pandas based on the task at hand

## Target Concepts

| Concept | Gloss |
|---------|-------|
| JSON Serialization | Records as text: JSON stores structured data as readable key value pairs, and JSON Lines (JSONL) stores one JSON object per line for streams and large datasets. |
| Filesystem Paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. |
| Workflow Form Choice | Explore in a notebook, automate once in a script, reuse reliably as a package with tests, or expose it as a service that other systems can call. |


## When to Use stdlib vs pandas

Recall the workflow-form choice: explore in a notebook, automate once in a script, reuse as a package, or expose a service. The same rule applies to libraries: **match the tool to the task**.

| Situation | stdlib (`csv`, `json`) | pandas |
|-----------|----------------------|--------|
| Read a few records in a script | ✅ Zero dependencies | Overkill |
| Analyze / filter / aggregate rows | Manual loops | ✅ One-liner |
| Stream large files row by row | ✅ Low memory | Loads everything |
| Quick EDA in a notebook | Tedious formatting | ✅ `.head()`, `.describe()` |

**Rule of thumb:** if you need to *transform* tabular data, reach for pandas. If you just need to *read/write* a handful of records in a lightweight script, stdlib is enough. Exploring in a notebook favours pandas; a script or package that should run anywhere with few dependencies often favours the stdlib.

## Setup

> 🧩 **Notebook, script, package or service.** The form choice from the video, now used to pick stdlib or pandas.

We'll use `pathlib.Path` (recall from `mini-path-safe-loader`) and explicit `encoding="utf-8"` throughout.

In [2]:
import csv
import json
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

# Temp directory so the lab leaves no mess
tmp = Path(tempfile.mkdtemp())
print(f"Working in: {tmp}")

Working in: C:\Users\molak\AppData\Local\Temp\tmp1yjo91lv


## Sample Data

A handful of chat-message records — the kind of data you'll see throughout this course.

In [ ]:
messages = [
    {"id": 1, "role": "user",      "text": "What is RAG?",              "tokens": 5},
    {"id": 2, "role": "assistant", "text": "Retrieval-Augmented Generation.", "tokens": 4},
    {"id": 3, "role": "user",      "text": "Explain with an example.",  "tokens": 6},
    {"id": 4, "role": "assistant", "text": "Sure — imagine a café FAQ bot.", "tokens": 9},
]

print(f"{len(messages)} records ready")

---

## 1 · CSV — Write & Read

### 1a · stdlib `csv`

In [ ]:
csv_path = tmp / "messages.csv"

# Write
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=messages[0].keys())
    writer.writeheader()
    writer.writerows(messages)

print(f"Wrote {csv_path.name}  ({csv_path.stat().st_size} bytes)")
print(csv_path.read_text(encoding="utf-8")[:200])

In [ ]:
# Read back with stdlib
with csv_path.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows_stdlib = list(reader)

print(f"Read {len(rows_stdlib)} rows")
print("First row:", rows_stdlib[0])
print()
print("⚠️  Notice: csv.DictReader returns ALL values as strings.")
print(f"   tokens type: {type(rows_stdlib[0]['tokens'])}")

### 1b · pandas

In [ ]:
# Write
df = pd.DataFrame(messages)
csv_pd_path = tmp / "messages_pd.csv"
df.to_csv(csv_pd_path, index=False, encoding="utf-8")
print(f"Wrote {csv_pd_path.name}")

# Read
df_read = pd.read_csv(csv_pd_path, encoding="utf-8")
print(f"\ntokens dtype: {df_read['tokens'].dtype}  ← pandas infers int automatically")
df_read

> **stdlib vs pandas for CSV:** stdlib keeps everything as strings and streams row by row (low memory). pandas auto-infers types and gives you DataFrames — ideal for analysis.

---

## 2 · JSON — Write & Read

Standard JSON stores the entire dataset as **one array**.

### 2a · stdlib `json`

> 🧩 **json serialization.** Records as text: JSON stores structured data as readable key-value pairs.
> Watch for the records to round-trip as a list of dicts.


In [ ]:
json_path = tmp / "messages.json"

# Write
json_path.write_text(
    json.dumps(messages, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Wrote {json_path.name}  ({json_path.stat().st_size} bytes)")

# Read
loaded = json.loads(json_path.read_text(encoding="utf-8"))
print(f"Read {len(loaded)} records")
print(f"tokens type: {type(loaded[0]['tokens'])}  ← json preserves int/str")

### 2b · pandas

In [ ]:
json_pd_path = tmp / "messages_pd.json"

# Write — orient="records" gives a flat list of objects
df.to_json(json_pd_path, orient="records", indent=2, force_ascii=False)

# Read
df_json = pd.read_json(json_pd_path, orient="records", encoding="utf-8")
df_json

---

## 3 · JSON Lines (JSONL) — Write & Read

JSON Lines stores **one JSON object per line** — no wrapping array, no commas between records. This makes it:

- **Appendable** — add a new record by appending one line
- **Streamable** — process huge files line by line
- **The standard format** for LLM fine-tuning datasets (OpenAI, etc.)

### 3a · stdlib

In [ ]:
jsonl_path = tmp / "messages.jsonl"

# Write — one json.dumps() per line
with jsonl_path.open("w", encoding="utf-8") as f:
    for msg in messages:
        f.write(json.dumps(msg, ensure_ascii=False) + "\n")

print(f"Wrote {jsonl_path.name}")
print(jsonl_path.read_text(encoding="utf-8"))

In [ ]:
# Read — one json.loads() per line
with jsonl_path.open(encoding="utf-8") as f:
    loaded_jsonl = [json.loads(line) for line in f if line.strip()]

print(f"Read {len(loaded_jsonl)} records from JSONL")
print("Last record:", loaded_jsonl[-1])

### 3b · pandas

In [ ]:
jsonl_pd_path = tmp / "messages_pd.jsonl"

# Write — lines=True produces JSON Lines
df.to_json(jsonl_pd_path, orient="records", lines=True, force_ascii=False)

# Read — lines=True reads JSON Lines
df_jsonl = pd.read_json(jsonl_pd_path, orient="records", lines=True, encoding="utf-8")
df_jsonl

---

## Format Comparison

Let's compare all three formats side by side.

In [ ]:
comparison = [
    "| Feature | CSV | JSON | JSON Lines |",
    "|---------|-----|------|------------|",
    "| Extension | `.csv` | `.json` | `.jsonl` |",
    "| Structure | Rows + header | One array of objects | One object per line |",
    "| Types preserved? | ❌ All strings | ✅ int, str, bool, null | ✅ int, str, bool, null |",
    "| Nested data? | ❌ Flat only | ✅ Yes | ✅ Yes |",
    "| Appendable? | ⚠️ Fragile | ❌ Must rewrite | ✅ Append one line |",
    "| Streamable? | ✅ Row by row | ❌ Parse whole file | ✅ Line by line |",
    "| LLM fine-tune format? | ❌ | ❌ | ✅ Standard |",
]

md("\n".join(comparison))

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tmp)
print(f"Cleaned up: {tmp}")

## 🎯 Summary

### Key Takeaways

1. **CSV** — great for flat tabular data, but loses type information (everything becomes a string in stdlib); pandas auto-infers types on read
2. **JSON** — preserves types and supports nested data, but you must read/write the entire file at once
3. **JSON Lines** — one object per line; appendable, streamable, and the standard format for LLM fine-tuning datasets
4. **Match the tool to the workflow form** — stdlib (`csv`, `json`) for lightweight scripts and packages; pandas for notebook exploration, analysis and transformation
5. **Always use `pathlib.Path` + `encoding="utf-8"`** — portable paths and explicit encoding prevent cross-platform bugs (recall from `mini-path-safe-loader`)